In [1]:
import numpy as np
from helpers import *
from implementations import * 


DATA_PATH = "./dataset/"

### Loading data

In [2]:
# Load data from CSV files
print("Loading data...")
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
print("Data loaded")

print("Negative class rate:", np.mean(y_train == -1))
print("Positive class rate:", np.mean(y_train == 1))

Loading data...
Data loaded
Negative class rate: 0.911697929205967
Positive class rate: 0.08830207079403295


There is a strong imbalance in the dataset. Only about 8.8 % of examples are positive.

A model that predicts every example as negative might get very high accuracy, but the positive-class F1 score would be very low. 

### Preprocessing of data

Before applying a model to our data, it first needs to be preprocessed. 

We start by performing one-hot encoding to address the problem of categorical features. A linear model treats categories  (e.g. $1, 2, 3, 4, 5$) as numbers, implying that category $5$ is "five times greater" than category $1$. One-hot encoding turns them into independent binary values, allowing  logistic regression weights ($w$) to learn a distinct coefficient for each specific category independently.

We then search for columns in the training set with a percentage of Nan values higher than 90%, because we assume that a too high percentage of missing values makes them irrelevant. These columns are removed in both the training and test data sets, because the model must be trained and evaluated using the same features.

In the remaining columns, Nan values are replaced by column medians. Column medians are chosen over column means, because medians are typically less sensitive to outliers than means. The medians are calculated using the training set only, then those same medians are used to fill missing values in both the training and the test set. This prevents data leakage from the test set into the training process.

The standardization parameters (mean, standard deviation) are calculated based on the training set. Then the features of both the training and the test sets are Z-normalized by removing the mean and dividing by the standard deviation. Again, only training statistics are used for both datasets to avoid influence of the test set on prepropressing and assuring consistency.

Finally, a column of 1s is added at the beginning of the training and test sets to account for the bias term (w0, intercept) in our model.

In [3]:
def preprocess_data(x_train, x_test, nan_threshold=0.90):
    """
    Perform preprocessing on training and testing datasets. 
    Includes:
     1. Removing columns with excessive Nan values.
     2. One-hot encoding of categorical features
     3. Replacing missing values with column medians (median imputation).
     4. Z-score standardization.
     5. Adding bias column of ones.
    
    Parameters:
    - x_train: array of raw training features 
    - x_test: array of raw testing features 
    - nan_threshold: column NaN percentage limit to keep (e.g. 0.90 = keep if <= 90% NaN)
    
    Returns:
    - x_train_final: filtered, imputed and scaled training set 
    - x_test_final: filtered, imputed and scaled test set 
    """
    print("--- Starting Preprocessing ---")
    
    # 1. Remove columns with high NaN ratios (> nan_threshold)
    nan_ratios_columns = np.isnan(x_train).mean(axis=0)
    columns_to_keep = nan_ratios_columns <= nan_threshold
    
    x_tr_filtered = x_train[:, columns_to_keep].copy()
    x_te_filtered = x_test[:, columns_to_keep].copy()

    nb_columns_removed = np.sum(~columns_to_keep) # The ~ operator performs a logical NOT on Boolean values --> False becomes True and vice vera
    percentage_nan_threshold = nan_threshold * 100

    print(f'Number of removed columns with >{percentage_nan_threshold:.0f}% NaNs: {nb_columns_removed}')
    print(f'Remaining features: {x_tr_filtered.shape[1]}')

    # 2. One-hot encode categorical features first (or right after filtering)
    x_tr_encoded, x_te_encoded = one_hot_encode_categorical(x_tr_filtered, x_te_filtered)
    print(f'Features shape after one-hot encoding: {x_tr_encoded.shape[1]}')
    
    # 3. Replace NaN values by column median values (operating on the encoded arrays)
    train_medians = np.nanmedian(x_tr_encoded, axis=0) 
    train_medians = np.nan_to_num(train_medians, nan=0.0) 
    
    for col_idx in range(x_tr_filtered.shape[1]):
        # Replaces missing values in training set with corresponding column median
        train_nan = np.isnan(x_tr_encoded[:, col_idx])
        x_tr_filtered[train_nan, col_idx] = train_medians[col_idx]
        
        # Replaces missing values in testing set with corresponding column median from training set 
        test_nan = np.isnan(x_te_encoded[:, col_idx])
        x_te_filtered[test_nan, col_idx] = train_medians[col_idx]
        
    print("Missing values replaced using medians from the training set.")
    
    # 3. Standardization (Z-score normalization)
    mean = np.mean(x_tr_encoded, axis=0)
    std = np.std(x_tr_encoded, axis=0)
    
    # Special case: division by zero --> std replaced by 1.0 for constant columns
    std[std == 0] = 1.0
    
    x_tr_normalized = (x_tr_encoded - mean) / std
    x_te_normalized = (x_te_encoded - mean) / std
    print("Features standardized (zero mean, unit variance).")
    
    # 4. Add a column of 1s at the beginning for the bias weight (w_0)
    bias_train = np.ones((x_tr_normalized.shape[0], 1))
    bias_test = np.ones((x_te_normalized.shape[0], 1))
    
    x_tr_final = np.hstack((bias_train, x_tr_normalized))
    x_te_final = np.hstack((bias_test, x_te_normalized))
    
    print(f"Final processed training shape: {x_tr_final.shape}")
    print(f"Final processed testing shape: {x_te_final.shape}")
    print("--- Preprocessing Complete ---\n")
    
    return x_tr_final, x_te_final

What we could do :
- Handle categorical variables : Categorical features might be mistakenly treated as continuous. Therefore, categorical columns should be converted into binary indicator variables if applicable.

### Possible values for the hypervariables

In [ ]:
# Hypervariables
nan_thresholds = [0.60, 0.70, 0.80, 0.85, 0.90, 0.95]
regularizers = np.arange(0.0001, 0.0011, 0.0001)
iterations = np.arange(100, 501, 100)
learning_rate = np.arange(0.3, 0.9, 0.1)
decision_thresholds = np.arange(0.10, 0.501, 0.01)

In [ ]:
nan_threshold = 0.70
regularizer = 0.0005
iterations = 300
learning_rate = 0.5


### 1. Experiment - Compare performance and choose threshold

### 1.1. Experiment A : Logistic regression on non-weighted datasets

In [ ]:
# 1. Split the original data BEFORE preprocessing
rng = np.random.default_rng(42)
indices = rng.permutation(len(y_train))

# 80% train, 20% validation
split = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx = indices[split:]

x_tr_raw = x_train[train_idx] 
x_val_raw = x_train[val_idx] 

y_tr = y_train[train_idx]  
y_val = y_train[val_idx]

# 2. Fit preprocessing on training data only, with a given Nan threshold 

# Evaluate different Nan thresholds
for nan_threshold in nan_thresholds: 
    x_tr, x_val = preprocess_data(
        x_tr_raw, x_val_raw, nan_threshold
    )

    # Convert labels from {-1, +1} to {0, 1}
    y_tr_bin = np.where(y_tr == -1, 0, 1).reshape(-1, 1)
    y_val_bin = (y_val == 1).astype(int).ravel()


    # 3. Calculate class weights
    negative_rate = np.mean(y_tr_bin.ravel() == 0)
    positive_rate = np.mean(y_tr_bin.ravel() == 1)

    weight_negative = 1.0 / (2 * negative_rate)
    weight_positive = 1.0 / (2 * positive_rate)

    print("Training negative rate:", negative_rate)
    print("Training positive rate:", positive_rate)
    print("Negative class weight:", weight_negative)
    print("Positive class weight:", weight_positive)


    # 4. Train the same regularized logistic regression
    w_init = np.zeros((x_tr.shape[1], 1))

    # Using non-weighted regularized logistic regression
    w_opt, loss = reg_logistic_regression(
        y_tr_bin,
        x_tr,
        regularizer,       
        w_init,
        iterations,          
        learning_rate        
    )

    # 5. Calculate validation probabilities
    probs = np.asarray(sigmoid(x_val @ w_opt)).ravel()

    # 6. Evaluate multiple decision thresholds
    results = []

    for threshold in decision_thresholds:
        pred = (probs >= threshold).astype(int)
        precision, recall, accuracy = compute_metrics(pred, y_val_bin)
        results.append((f1, threshold, precision, recall, accuracy))

    # Show the best thresholds by validation F1
    results.sort(reverse=True)

    print("\nBest validation thresholds:")
    print("F1       Threshold  Precision  Recall  Accuracy")

    for f1, threshold, precision, recall, accuracy in results[:10]:
        print(
            f"{f1:.4f}   {threshold:.2f}       "
            f"{precision:.4f}     {recall:.4f}  {accuracy:.4f}"
        )


--- Starting Preprocessing ---
Number of removed columns with >90% NaNs: 99
Remaining features: 222
Smart-detected 122 true categorical columns out of 222.
Features shape after one-hot encoding: 637
Missing values replaced using medians from the training set.
Features standardized (zero mean, unit variance).
Final processed training shape: (262508, 638)
Final processed testing shape: (65627, 638)
--- Preprocessing Complete ---

Training negative rate: 0.9118122114373657
Training positive rate: 0.08818778856263428
Negative class weight: 0.5483585257229756
Positive class weight: 5.669719222462203

Best validation thresholds:
F1       Threshold  Precision  Recall  Accuracy
0.0000   0.80       0.0000     0.0000  0.9112
0.0000   0.79       0.0000     0.0000  0.9112
0.0000   0.78       0.0000     0.0000  0.9112
0.0000   0.77       0.0000     0.0000  0.9112
0.0000   0.76       0.0000     0.0000  0.9112
0.0000   0.75       0.0000     0.0000  0.9112
0.0000   0.74       0.0000     0.0000  0.9112

### 1.2. Experiment B : Weighted logistic regression

In [ ]:
# 1. Split the original data BEFORE preprocessing
rng = np.random.default_rng(42)
indices = rng.permutation(len(y_train))

# 80% train, 20% validation
split = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx = indices[split:]

x_tr_raw = x_train[train_idx] 
x_val_raw = x_train[val_idx] 

y_tr = y_train[train_idx]  
y_val = y_train[val_idx]

# 2. Fit preprocessing on training data only, with a given Nan threshold 

# Evaluate different Nan thresholds
for nan_threshold in nan_thresholds:
    x_tr, x_val = preprocess_data(
        x_tr_raw, x_val_raw, nan_threshold=0.90
    )

    # Convert labels from {-1, +1} to {0, 1}
    y_tr_bin = np.where(y_tr == -1, 0, 1).reshape(-1, 1)
    y_val_bin = (y_val == 1).astype(int).ravel()


    # 3. Calculate class weights
    negative_rate = np.mean(y_tr_bin.ravel() == 0)
    positive_rate = np.mean(y_tr_bin.ravel() == 1)

    weight_negative = 1.0 / (2 * negative_rate)
    weight_positive = 1.0 / (2 * positive_rate)

    print("Training negative rate:", negative_rate)
    print("Training positive rate:", positive_rate)
    print("Negative class weight:", weight_negative)
    print("Positive class weight:", weight_positive)


    # 4. Train the same regularized logistic regression
    w_init = np.zeros((x_tr.shape[1], 1))

    # Using weighted regularized logistic regression
    w_opt, loss = weighted_reg_logistic_regression(
        y_tr_bin,
        x_tr,
        regularizer,       # regularization
        w_init,
        iterations,          # iterations
        learning_rate           # learning rate
    )

    # 5. Calculate validation probabilities
    probs = np.asarray(sigmoid(x_val @ w_opt)).ravel()

    # 6. Evaluate multiple decision thresholds
    thresholds = np.arange(0.10, 0.201, 0.01)
    results = []

    for threshold in thresholds:
        pred = (probs >= threshold).astype(int)
        precision, recall, accuracy = compute_metrics(pred, y_val_bin)
        results.append((f1, threshold, precision, recall, accuracy))

    # Show the best thresholds by validation F1
    results.sort(reverse=True)

    print("\nBest validation thresholds:")
    print("F1       Threshold  Precision  Recall  Accuracy")

    for f1, threshold, precision, recall, accuracy in results[:10]:
        print(
            f"{f1:.4f}   {threshold:.2f}       "
            f"{precision:.4f}     {recall:.4f}  {accuracy:.4f}"
        )

--- Starting Preprocessing ---
Number of removed columns with >90% NaNs: 99
Remaining features: 222
Smart-detected 122 true categorical columns out of 222.
Features shape after one-hot encoding: 637
Missing values replaced using medians from the training set.
Features standardized (zero mean, unit variance).
Final processed training shape: (262508, 638)
Final processed testing shape: (65627, 638)
--- Preprocessing Complete ---

Training negative rate: 0.9118122114373657
Training positive rate: 0.08818778856263428
Negative class weight: 0.5483585257229756
Positive class weight: 5.669719222462203

Best validation thresholds:
F1       Threshold  Precision  Recall  Accuracy
0.0000   0.20       0.0000     0.0000  0.9112
0.0000   0.19       0.0000     0.0000  0.9112
0.0000   0.18       0.0000     0.0000  0.9112
0.0000   0.17       0.0000     0.0000  0.9112
0.0000   0.16       0.0000     0.0000  0.9112
0.0000   0.15       0.0000     0.0000  0.9112
0.0000   0.14       0.0000     0.0000  0.9112

### 2. 5-fold cross-validation

In [ ]:
# Convert original labels {-1, +1} to binary labels {0, 1}.
y_all_binary = np.where(
    np.asarray(y_train).ravel() == -1, 0, 1
)

X_raw = np.asarray(x_train)
n_folds = 5

folds = make_stratified_folds(
    y_all_binary, n_folds=n_folds, seed=42
)

# Each row receives one prediction from a model that did not train on it.
oof_normal = np.zeros(len(y_all_binary))
oof_weighted = np.zeros(len(y_all_binary))

for fold_num, val_idx in enumerate(folds, start=1):
    print(f"\n========== Fold {fold_num}/{n_folds} ==========")

    train_idx = np.setdiff1d(
        np.arange(len(y_all_binary)),
        val_idx,
        assume_unique=True
    )

    X_fold_train_raw = X_raw[train_idx]
    X_fold_val_raw = X_raw[val_idx]

    y_fold_train = y_all_binary[train_idx]
    y_fold_val = y_all_binary[val_idx]

    # Fit preprocessing only on this fold's training data.
    X_fold_train, X_fold_val = preprocess_data(
        X_fold_train_raw,
        X_fold_val_raw,
        nan_threshold=nan_threshold
    )

    print("Training normal SGD...")
    w_normal = train_logistic_sgd(
        y_fold_train,
        X_fold_train,
        lambda_=0.0005,
        learning_rate,
        epochs=12,
        batch_size=2048,
        weighted=False,
        seed=100 + fold_num
    )

    print("Training weighted SGD...")
    w_weighted = train_logistic_sgd(
        y_fold_train,
        X_fold_train,
        lambda_=0.0005,
        learning_rate,
        epochs=12,
        batch_size=2048,
        weighted=True,
        seed=100 + fold_num
    )

    # Save probabilities for the held-out fold.
    oof_normal[val_idx] = sigmoid_safe(
        X_fold_val @ w_normal
    ).ravel()

    oof_weighted[val_idx] = sigmoid_safe(
        X_fold_val @ w_weighted
    ).ravel()

    print(
        f"Fold {fold_num} complete. "
        f"Validation rows: {len(val_idx):,}"
    )

print("\nCross-validation predictions are ready.")


========== Fold 1/5 ==========
--- Starting Preprocessing ---
Number of removed columns with >90% NaNs: 99
Remaining features: 222
Smart-detected 121 true categorical columns out of 222.
Features shape after one-hot encoding: 642
Missing values replaced using medians from the training set.
Features standardized (zero mean, unit variance).
Final processed training shape: (262508, 643)
Final processed testing shape: (65627, 643)
--- Preprocessing Complete ---

Training normal SGD...
Training weighted SGD...
Fold 1 complete. Validation rows: 65,627

========== Fold 2/5 ==========
--- Starting Preprocessing ---
Number of removed columns with >90% NaNs: 99
Remaining features: 222
Smart-detected 121 true categorical columns out of 222.
Features shape after one-hot encoding: 641
Missing values replaced using medians from the training set.
Features standardized (zero mean, unit variance).
Final processed training shape: (262508, 642)
Final processed testing shape: (65627, 642)
--- Preprocessi

### 4. Compare F1, precision, and recall

In [ ]:
normal_results = find_best_threshold(
    y_all_binary, oof_normal
)
weighted_results = find_best_threshold(
    y_all_binary, oof_weighted
)

print("NORMAL SGD — best thresholds")
print("F1       Threshold  Precision  Recall  Accuracy")
for row in normal_results[:10]:
    print(
        f'{row["F1"]:.4f}   '
        f'{row["Threshold"]:.2f}       '
        f'{row["Precision"]:.4f}     '
        f'{row["Recall"]:.4f}  '
        f'{row["Accuracy"]:.4f}'
    )

print("\nWEIGHTED SGD — best thresholds")
print("F1       Threshold  Precision  Recall  Accuracy")
for row in weighted_results[:10]:
    print(
        f'{row["F1"]:.4f}   '
        f'{row["Threshold"]:.2f}       '
        f'{row["Precision"]:.4f}     '
        f'{row["Recall"]:.4f}  '
        f'{row["Accuracy"]:.4f}'
    )

NORMAL SGD — best thresholds
F1       Threshold  Precision  Recall  Accuracy
0.0000   0.05       0.0000     0.0000  0.9117
0.0000   0.06       0.0000     0.0000  0.9117
0.0000   0.07       0.0000     0.0000  0.9117
0.0000   0.08       0.0000     0.0000  0.9117
0.0000   0.09       0.0000     0.0000  0.9117
0.0000   0.10       0.0000     0.0000  0.9117
0.0000   0.11       0.0000     0.0000  0.9117
0.0000   0.12       0.0000     0.0000  0.9117
0.0000   0.13       0.0000     0.0000  0.9117
0.0000   0.14       0.0000     0.0000  0.9117

WEIGHTED SGD — best thresholds
F1       Threshold  Precision  Recall  Accuracy
0.0000   0.05       0.0000     0.0000  0.9117
0.0000   0.06       0.0000     0.0000  0.9117
0.0000   0.07       0.0000     0.0000  0.9117
0.0000   0.08       0.0000     0.0000  0.9117
0.0000   0.09       0.0000     0.0000  0.9117
0.0000   0.10       0.0000     0.0000  0.9117
0.0000   0.11       0.0000     0.0000  0.9117
0.0000   0.12       0.0000     0.0000  0.9117
0.0000   0.13  

### 5. Select the winner and train on all labeled data

In [10]:
# Select the model with the highest out-of-fold F1.
best_normal = normal_results[0]
best_weighted = weighted_results[0]

if best_weighted["F1"] > best_normal["F1"]:
    chosen_model = "weighted"
    best_result = best_weighted
else:
    chosen_model = "normal"
    best_result = best_normal

best_threshold = best_result["Threshold"]

print("Selected model:", chosen_model)
print("Selected OOF threshold:", best_threshold)
print("Best OOF F1:", round(best_result["F1"], 4))

# Fit preprocessing on all labeled training rows.
X_full, X_test_proc = preprocess_data(
    x_train, x_test, nan_threshold=0.90
)

# Retrain the selected model on all labeled examples.
w_final = train_logistic_sgd(
    y_all_binary,
    X_full,
    lambda_=0.0005,
    learning_rate=0.05,
    epochs=12,
    batch_size=2048,
    weighted=(chosen_model == "weighted"),
    seed=42
)

# Predict probabilities, then convert back to {-1, +1}.
test_probabilities = sigmoid_safe(
    X_test_proc @ w_final
).ravel()

test_predictions = np.where(
    test_probabilities >= best_threshold, 1, -1
)

print("Positive test predictions:", np.sum(test_predictions == 1))
print("Negative test predictions:", np.sum(test_predictions == -1))

create_csv_submission(
    test_ids,
    test_predictions,
    "sgd_logistic_submission.csv"
)

print("Submission file created: sgd_logistic_submission.csv")

Selected model: normal
Selected OOF threshold: 0.05
Best OOF F1: 0.0
--- Starting Preprocessing ---
Number of removed columns with >90% NaNs: 99
Remaining features: 222
Smart-detected 121 true categorical columns out of 222.
Features shape after one-hot encoding: 644
Missing values replaced using medians from the training set.
Features standardized (zero mean, unit variance).
Final processed training shape: (328135, 645)
Final processed testing shape: (109379, 645)
--- Preprocessing Complete ---



KeyboardInterrupt: 

### Determination of model overfits or underfits

Underfitting
- Symptoms : 
    - High training loss / low training F1
    - High validation loss / low validation F1
- Causes : 
    - Model too simple
    - Regularization (lambda) too high
    - Not enough training iterations

Overfitting
- Symptoms : 
    - Very low training loss / high training F1
    - High validation loss / low validation F1
- Causes :
    - Model too complex
    - No regularization or too small
    - Too many noisy features

